# Cell-jamming & parameter-gradient migration — analysis & visualisation

This notebook reads the **archived simulation histories** written by
`jamming_gradient.py` (`outputs/jamming/history.hf5` and
`outputs/gradient/history.hf5`) and produces everything that used to be baked into
the run script:

* a colour-coded **GIF** (faces tinted by preferred perimeter, the migrating cell's
  junctions highlighted) and **still frames** per scenario,
* the migrating-cell **trace** (`migrating_trace.csv`) and its plots — displacement
  over time (with the jamming-transition marker), velocity vs x-position, and
  preferred perimeter over time,
* cell-**circularity** analyses (mean circularity over time for jamming; circularity
  binned along x for the gradient).

It only *reads* the archives, so you can re-analyse without re-simulating — and
re-run `jamming_gradient.py` without disturbing earlier analysis.

Run from the repo's **`vivarium-tyssue` conda env** (needs ImageMagick `magick`
on PATH for the GIFs).

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Simulation config is the single source of truth — import it from the sim script.
import jamming_gradient as sim
from jamming_gradient import OUT_DIR, COORDS, PP_RANGE, JAM_TRIGGER

sys.path.insert(0, str(sim.REPO))   # so vivarium_tyssue.draw / tyssue import cleanly
from tyssue.core.history import HistoryHdf5

# --- visualisation / analysis constants (were in the old monolithic script) ---
NUM_GIF_FRAMES = 200
N_STILLS = 5
GIF_DPI = 110
FIG_DPI = 300
LINE_COLOR = "#0072B2"     # Okabe-Ito blue
EVENT_COLOR = "#D55E00"    # Okabe-Ito vermillion (jamming-transition marker)

print("archives:", OUT_DIR)

## Loading an archived history

Same helper as the other experiment notebooks: reopen the archive with
`HistoryHdf5.from_archive` for drawing, read the full stacked dataframes from the
HDF5 store for analysis, and rebuild each retrieved frame from those stacked tables
(restoring the per-frame topological index from the `vert` / `edge` / `face`
columns) so `SheetGeometry.update_all` and `.loc`-based drawing work.

In [ ]:
class LoadedHistory:
    # Reopened archive with the same surface as an in-memory tyssue History:
    # .time_stamps, .retrieve(t) (drawing), .datasets (full stacked dfs, analysis).
    def __init__(self, path):
        self._h = HistoryHdf5.from_archive(str(path))
        self._sheet = self._h.sheet
        with pd.HDFStore(str(path), "r") as store:
            self.datasets = {k.strip("/"): store.select(k) for k in store.keys()}
        self._times = np.array(sorted(self.datasets["vert"]["time"].unique()))

    @property
    def time_stamps(self):
        return self._times

    def retrieve(self, t):
        # Rebuild a sheet at the nearest recorded time from the stacked datasets,
        # restoring each element's per-frame index from its vert/edge/face column
        # (HistoryHdf5.retrieve skips this, leaving srce/trgt misaligned).
        t = self._times[int(np.argmin(np.abs(self._times - t)))]
        sheet_datasets = {}
        for elem, df in self.datasets.items():
            sub = df[df["time"] == t]
            if elem in sub.columns:
                sub = sub.set_index(elem)
                sub.index.name = elem
            sheet_datasets[elem] = sub
        sheet = type(self._sheet)(f"{self._sheet.identifier}_{t:04.3f}",
                                  sheet_datasets, self._sheet.specs)
        sheet.coords = self._sheet.coords
        return sheet

    def update_datasets(self):
        pass

    def __getattr__(self, name):
        return getattr(self._h, name)   # e.g. .sheet, used by tyssue.create_gif

## Visualisation — GIF + stills

Faces tinted by `prefered_perimeter` (Reds colour bar over `PP_RANGE`), the
migrating cell's junctions highlighted cyan. Identical to the old `save_gif` /
`save_stills`.

In [ ]:
def _face_edge_specs(pp_range):
    # The notebook's face_param + migrating-cell edge draw kwds.
    from vivarium_tyssue.draw import face_param_kwds, migrating_cell_edge_kwds
    face = face_param_kwds("prefered_perimeter", color_range=pp_range)["face"]
    edge = migrating_cell_edge_kwds(
        highlight_color="cyan", highlight_alpha=0.6,
        base_color="black", base_alpha=0.8, width=1.5,
    )["edge"]
    return face, edge


def save_gif(history, out_path: Path, pp_range):
    from tyssue import config
    from tyssue.draw import create_gif
    face, edge = _face_edge_specs(pp_range)
    ds = config.draw.sheet_spec()
    ds["face"].update(face)
    ds["edge"].update(edge)
    ds["axis"].update({
        "color_bar": True, "color_bar_cmap": "Reds", "color_bar_range": tuple(pp_range),
        "color_bar_label": "prefered perimeter", "color_bar_target": "face",
    })
    create_gif(history, str(out_path), coords=COORDS, num_frames=NUM_GIF_FRAMES, dpi=GIF_DPI, **ds)


def save_stills(history, out_dir: Path, pp_range):
    from tyssue.draw import sheet_view
    from tyssue.geometry.sheet_geometry import SheetGeometry
    face, edge = _face_edge_specs(pp_range)
    face_color_fn, edge_color_fn = face["color"], edge["color"]
    times = list(history.time_stamps)
    if not times:
        return
    for frac in np.linspace(0.0, 1.0, N_STILLS):
        t = times[int(round(frac * (len(times) - 1)))]
        sheet = history.retrieve(t)
        SheetGeometry.update_all(sheet)
        fig, ax = plt.subplots(figsize=(4.2, 4.2))
        sheet_view(
            sheet, coords=COORDS, ax=ax,
            face={"visible": True, "color": face_color_fn(sheet), "alpha": 0.9},
            edge={"visible": True, "color": edge_color_fn(sheet), "width": 1.5},
        )
        ax.set_title(f"t = {float(t):.1f}", fontsize=9)
        ax.set_aspect("equal")
        fig.savefig(out_dir / f"still_t{float(t):05.1f}.png", dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)

## Analysis — migrating-cell trace & circularity

`migrating_trace` reads the migrating cell (largest `migration_strength`) straight
from the stacked face dataframe; circularity is `4πA / P²` per alive face. Unchanged
from the original script.

In [ ]:
def migrating_trace(history) -> pd.DataFrame:
    # Per-timepoint state of the migrating cell: time, x, y (centroid),
    # prefered_perimeter, displacement (from t0), speed (instantaneous |velocity|).
    face = history.datasets["face"]
    mig = face[face["migration_strength"] > 0].sort_values("time").drop_duplicates("time", keep="first")
    mig = mig.reset_index(drop=True)
    x0, y0 = float(mig.loc[0, "x"]), float(mig.loc[0, "y"])
    out = pd.DataFrame({
        "time": mig["time"].astype(float).to_numpy(),
        "x": mig["x"].astype(float).to_numpy(),
        "y": mig["y"].astype(float).to_numpy(),
        "prefered_perimeter": mig["prefered_perimeter"].astype(float).to_numpy(),
    })
    out["displacement"] = np.hypot(out["x"] - x0, out["y"] - y0)
    dt = out["time"].diff()
    dist = np.hypot(out["x"].diff(), out["y"].diff())
    out["speed"] = dist / dt
    return out


def _face_circularity(history):
    # All alive faces across all frames with a circularity column (4*pi*A / P**2).
    face = history.datasets["face"]
    df = face[(face["is_alive"] > 0) & (face["perimeter"] > 0) & (face["area"] > 0)].copy()
    df["circularity"] = 4.0 * np.pi * df["area"].astype(float) / df["perimeter"].astype(float) ** 2
    return df


def circularity_over_time(history) -> pd.DataFrame:
    # Mean cell circularity at each timepoint. Columns: time, circularity.
    df = _face_circularity(history)
    return df.groupby("time")["circularity"].mean().reset_index()


def circularity_along_x(history, nbins: int = 30) -> pd.DataFrame:
    # Mean cell circularity binned by face x-position, pooled over all timepoints.
    df = _face_circularity(history)
    x = df["x"].astype(float)
    bins = np.linspace(x.min(), x.max(), nbins + 1)
    df["xbin"] = pd.cut(x, bins, include_lowest=True)
    g = df.groupby("xbin", observed=True)["circularity"].agg(["mean", "std", "count"]).reset_index()
    g["xcenter"] = g["xbin"].apply(lambda b: float(b.mid))
    return g[["xcenter", "mean", "std", "count"]].rename(columns={"mean": "circularity"})

## Analysis plots

In [ ]:
def plot_displacement(trace, out_path: Path, title, jamming_time=None):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    ax.plot(trace["time"], trace["displacement"], "-", color=LINE_COLOR, linewidth=2,
            label="migrating cell")
    if jamming_time is not None:
        ax.axvline(jamming_time, linestyle=":", color=EVENT_COLOR, linewidth=1.8)
        ax.text(jamming_time, ax.get_ylim()[1], "  jamming", color=EVENT_COLOR,
                va="top", ha="left", fontsize=9)
    ax.set_xlabel("time")
    ax.set_ylabel("displacement from initial position")
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    fig.tight_layout(); fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()


def plot_velocity_vs_x(trace, out_path: Path, title):
    t = trace.dropna(subset=["speed"])
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    sc = ax.scatter(t["x"], t["speed"], c=t["time"], cmap="viridis", s=14)
    ax.plot(t["x"], t["speed"], "-", color="0.7", linewidth=0.7, zorder=0)
    cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04); cb.set_label("time", fontsize=9)
    ax.set_xlabel("migrating cell x-position")
    ax.set_ylabel("instantaneous speed")
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    fig.tight_layout(); fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()


def plot_pref_perimeter_vs_time(trace, out_path: Path, title):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    ax.plot(trace["time"], trace["prefered_perimeter"], "-", color=LINE_COLOR, linewidth=2)
    ax.set_xlabel("time")
    ax.set_ylabel("prefered perimeter (migrating cell)")
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    fig.tight_layout(); fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()


def plot_circularity_over_time(circ, out_path: Path, title, jamming_time=None):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    ax.plot(circ["time"], circ["circularity"], "-", color=LINE_COLOR, linewidth=2)
    if jamming_time is not None:
        ax.axvline(jamming_time, linestyle=":", color=EVENT_COLOR, linewidth=1.8)
        ax.text(jamming_time, ax.get_ylim()[1], "  jamming", color=EVENT_COLOR,
                va="top", ha="left", fontsize=9)
    ax.set_xlabel("time")
    ax.set_ylabel("mean cell circularity  (4piA / P^2)")
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    fig.tight_layout(); fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()


def plot_circularity_along_x(circ, out_path: Path, title):
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    m = circ["circularity"].to_numpy(); s = circ["std"].to_numpy(); x = circ["xcenter"].to_numpy()
    ax.fill_between(x, m - s, m + s, color=LINE_COLOR, alpha=0.15, linewidth=0)
    ax.plot(x, m, "-o", color=LINE_COLOR, linewidth=2, markersize=4)
    ax.set_xlabel("face x-position")
    ax.set_ylabel("mean cell circularity  (4piA / P^2)")
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    fig.tight_layout(); fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.show()

## Jamming scenario

Set `MAKE_GIFS = False` to skip the (slow) animation while iterating on the plots.

In [ ]:
MAKE_GIFS = True

jam_dir = OUT_DIR / "jamming"
jam_path = jam_dir / "history.hf5"
if not jam_path.exists():
    print("no jamming archive — run `python jamming_gradient.py` first")
else:
    hist_j = LoadedHistory(jam_path)
    if MAKE_GIFS:
        save_gif(hist_j, jam_dir / "jamming.gif", PP_RANGE)
    save_stills(hist_j, jam_dir, PP_RANGE)

    trace_j = migrating_trace(hist_j)
    trace_j.to_csv(jam_dir / "migrating_trace.csv", index=False)
    plot_displacement(trace_j, jam_dir / "migrating_displacement.png",
                      "Jamming: migrating-cell displacement", jamming_time=JAM_TRIGGER)

    circ_j = circularity_over_time(hist_j)
    circ_j.to_csv(jam_dir / "circularity_over_time.csv", index=False)
    plot_circularity_over_time(circ_j, jam_dir / "circularity_over_time.png",
                               "Jamming: mean cell circularity over time", jamming_time=JAM_TRIGGER)
    print(f"[jamming] final displacement = {trace_j['displacement'].iloc[-1]:.3f}")

## Gradient scenario

In [ ]:
grad_dir = OUT_DIR / "gradient"
grad_path = grad_dir / "history.hf5"
if not grad_path.exists():
    print("no gradient archive — run `python jamming_gradient.py` first")
else:
    hist_g = LoadedHistory(grad_path)
    if MAKE_GIFS:
        save_gif(hist_g, grad_dir / "gradient.gif", PP_RANGE)
    save_stills(hist_g, grad_dir, PP_RANGE)

    trace_g = migrating_trace(hist_g)
    trace_g.to_csv(grad_dir / "migrating_trace.csv", index=False)
    plot_displacement(trace_g, grad_dir / "migrating_displacement.png",
                      "Gradient: migrating-cell displacement")
    plot_velocity_vs_x(trace_g, grad_dir / "migrating_velocity_vs_x.png",
                       "Gradient: migrating-cell velocity vs x-position")
    plot_pref_perimeter_vs_time(trace_g, grad_dir / "prefered_perimeter_vs_time.png",
                                "Gradient: migrating-cell prefered perimeter over time")

    circ_gx = circularity_along_x(hist_g)
    circ_gx.to_csv(grad_dir / "circularity_along_x.csv", index=False)
    plot_circularity_along_x(circ_gx, grad_dir / "circularity_along_x.png",
                             "Gradient: mean cell circularity along x (all timepoints)")
    print(f"[gradient] final displacement = {trace_g['displacement'].iloc[-1]:.3f}")